In [ ]:
from importlib import import_module
from pathlib import Path
import json
import os
import random
import sys
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import EntryNotFoundError

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src/steps/04_evaluation/evaluator.py').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Open this notebook from inside the project.')
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env', override=False)
DATA_DIR = PROJECT_ROOT / '.data'
CURATED_DIR = DATA_DIR / 'curated'
TEST_FILE = CURATED_DIR / 'test.jsonl'

if TEST_FILE.is_file():
    print(f'Using existing curated test split: {TEST_FILE}')
else:
    dataset_repo = os.getenv('HF_DATASET_REPO', 'md-zain/resume-parser-dataset').strip()
    if dataset_repo.count('/') != 1:
        raise ValueError('HF_DATASET_REPO must be formatted as owner/dataset-name.')
    revision = os.getenv('HF_DATASET_REVISION', 'main').strip() or 'main'
    token = os.getenv('HF_TOKEN') or None
    if token and token.startswith(('hf_your_', 'hf_placeholder_')):
        token = None
    print(f'Curated test split not found locally; downloading it from {dataset_repo}...')
    try:
        hf_hub_download(
            repo_id=dataset_repo, repo_type='dataset', revision=revision,
            filename='curated/test.jsonl', local_dir=str(DATA_DIR), token=token,
        )
    except EntryNotFoundError:
        # Support the earlier dataset layout where the split files were at the repo root.
        hf_hub_download(
            repo_id=dataset_repo, repo_type='dataset', revision=revision,
            filename='test.jsonl', local_dir=str(CURATED_DIR), token=token,
        )
    if not TEST_FILE.is_file():
        raise FileNotFoundError(f'Download finished but the test split is missing: {TEST_FILE}')
    print(f'Downloaded curated test split to: {TEST_FILE}')

evaluation = import_module('src.steps.04_evaluation')
EvaluationItem = evaluation.EvaluationItem
Evaluator = evaluation.Evaluator
hf_inference = evaluation.hf_inference
ollama_inference = evaluation.ollama_inference
openrouter_inference = evaluation.openai_inference
report_results = evaluation.report_results

SAMPLE_SIZE, SEED, WORKERS = 5, 42, 1


In [2]:
def load_items(limit=None, seed=42):
    with TEST_FILE.open(encoding='utf-8') as handle:
        records = [json.loads(line) for line in handle if line.strip()]
    available = len(records)
    if limit is not None:
        if limit < 1:
            raise ValueError('limit must be positive or None.')
        records = random.Random(seed).sample(records, min(limit, available))

    items = []
    for record in records:
        items.append(EvaluationItem(
            resume_id=record['resume_id'],
            category=record['category'],
            resume_text=record['resume_text'],
            target_json=record['target_json'],
        ))
    return items, available

In [ ]:
items, available_pairs = load_items(SAMPLE_SIZE, SEED)
print(f'Available held-out test resumes: {available_pairs:,}')
print(f'Loaded for this test: {len(items)}')

In [ ]:
evaluator = Evaluator(items)
reports = {}

## Optional: Hugging Face model evaluation

Use this section only if you also want to evaluate local Transformers models from the Hub.

In [ ]:
HUGGINGFACE_MODELS = [
    'google/gemma-3-270m-it',
    'HuggingFaceTB/SmolLM2-360M-Instruct',
]

for model_name in HUGGINGFACE_MODELS:
    reports[f'huggingface:{model_name}'] = evaluator.evaluate(
        callback=hf_inference(model_name),
        workers=WORKERS,
    )


## Ollama — local model evaluation

Set the model names to models already available in your local Ollama installation, then run this section.

In [ ]:
OLLAMA_MODELS = ['llama3.2', 'gemma3:270m']
for model_name in OLLAMA_MODELS:
    reports[f'ollama:{model_name}'] = evaluator.evaluate(
        callback=ollama_inference(model_name),
        workers=WORKERS,
    )


## OpenRouter — hosted model evaluation

Set `OPENROUTER_API_KEY` in `.env`, then choose OpenRouter model IDs and run this section.

In [ ]:
OPENROUTER_MODELS = [
    'openai/gpt-6-luna',
]

for model_name in OPENROUTER_MODELS:
    reports[f'openrouter:{model_name}'] = evaluator.evaluate(
        callback=openrouter_inference(model_name),
        workers=WORKERS,
    )

## Combined comparison

Run this after the provider sections you want to evaluate to plot and compare their reports.

In [ ]:
# Display model charts and the metrics dashboard (nothing is saved yet).
summary = report_results(reports)


In [ ]:
# Save charts and metrics JSON under ./results/ when you are ready.
saved_to = summary.save()
saved_to